In [1]:
# ===== 1-Config + Tokenizer + REPEAT_S2 search =====
import os, json, re, gc, torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

MODEL_NAME  = '/root/autodl-tmp/employee_code/base_model'
EVAL_JSON   = '/root/autodl-tmp/civil_ft/data/eval_civil_code_questions.json'
S1_DIR      = '/root/autodl-tmp/civil_ft/checkpoints'
S2_DIR      = '/root/autodl-tmp/civil_ft/stage2_aligned'
RESULTS_DIR = '/root/autodl-tmp/civil_ft/results'
CONFIG_FILE = '/root/autodl-tmp/civil_ft/results/.s2_config.json'
for d in [S2_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)
LR_S2 = 2e-5; BS = 2; GAS = 8
EVAL_START, EVAL_END = 250, 530
FMT_THRESHOLD = 97

TOP5_S2 = '你是一个专业的中国法律检索专家。\n\n你的任务不是作答，而是为后续数据库检索生成候选法条。\n请根据用户案情，输出最相关的5条《中华人民共和国民法典》法条编号，按相关性从高到低排序。\n\n硬性要求：\n1. 必须恰好输出5条；\n2. 每条都必须是"《法律名称》第XXX条"；\n3. 不要输出解释、分析、理由、序号或其他任何文字；\n4. 即使不确定，也必须给出5条最可能的候选法条；\n5. 优先输出最核心、最直接适用的法条；\n6. 只输出 JSON，不要输出其他内容。\n\n输出格式：\n{"articles": ["《法律名称》第XXX条", ...]}'

RS2=[
 ('分期车第二年保险没在分期公司买，他们有权扣车么？',{'articles':['《中华人民共和国民法典》第四百二十八条','《中华人民共和国民法典》第三百八十六条','《中华人民共和国民法典》第四百零一条','《中华人民共和国民法典》第五百零九条','《中华人民共和国民法典》第五百七十七条']}),
 ('借钱给朋友，朋友不还怎么办？有录音和转账记录。',{'articles':['《中华人民共和国民法典》第六百七十条','《中华人民共和国民法典》第六百六十七条','《中华人民共和国民法典》第六百七十五条','《中华人民共和国民法典》第六百七十六条','《中华人民共和国民法典》第五百七十七条']}),
 ('我早上手机在教室丢了，去学校看监控被拒绝了。',{'articles':['《中华人民共和国民法典》第三百一十二条','《中华人民共和国民法典》第三百一十四条','《中华人民共和国民法典》第三百一十六条','《中华人民共和国民法典》第一百七十九条','《中华人民共和国民法典》第一千一百六十五条']}),
 ('我的父母想把他们的安置房送给孙子，没有房产证。',{'articles':['《中华人民共和国民法典》第二百零九条','《中华人民共和国民法典》第二百一十四条','《中华人民共和国民法典》第六百五十七条','《中华人民共和国民法典》第六百五十九条','《中华人民共和国民法典》第二百一十五条']}),
 ('去别人家鱼塘钓鱼被抓了，但一条鱼都没钓到。',{'articles':['《中华人民共和国民法典》第二百三十九条','《中华人民共和国民法典》第二百三十五条','《中华人民共和国民法典》第二百三十八条','《中华人民共和国民法典》第一百七十九条','《中华人民共和国民法典》第一千一百六十五条']}),
 ('大学生欠了很多贷款，没钱怎么办？',{'articles':['《中华人民共和国民法典》第六百七十五条','《中华人民共和国民法典》第六百六十七条','《中华人民共和国民法典》第六百七十六条','《中华人民共和国民法典》第五百七十七条','《中华人民共和国民法典》第五百八十五条']}),
 ('外祖父母过世后房产子女平分，孙辈没有继承权吧。',{'articles':['《中华人民共和国民法典》第一千一百二十七条','《中华人民共和国民法典》第一千一百二十二条','《中华人民共和国民法典》第一千一百二十三条','《中华人民共和国民法典》第一千一百二十八条','《中华人民共和国民法典》第一千一百三十条']}),
 ('父亲去世银行有存款怎么取？',{'articles':['《中华人民共和国民法典》第一千一百二十二条','《中华人民共和国民法典》第一千一百二十三条','《中华人民共和国民法典》第一千一百二十七条','《中华人民共和国民法典》第一千一百三十条','《中华人民共和国民法典》第一千一百四十五条']}),
 ('结婚和父母住，拆迁给二套房怎么分。',{'articles':['《中华人民共和国民法典》第一千一百二十七条','《中华人民共和国民法典》第二百零九条','《中华人民共和国民法典》第二百四十三条','《中华人民共和国民法典》第三百零一条','《中华人民共和国民法典》第一千零六十二条']}),
 ('买了二手房过了户受法律保护吗。',{'articles':['《中华人民共和国民法典》第二百零九条','《中华人民共和国民法典》第二百一十四条','《中华人民共和国民法典》第四百零六条','《中华人民共和国民法典》第五百零九条','《中华人民共和国民法典》第五百七十七条']})]
S2_QUESTIONS = {q for q, _ in RS2}

# Tokenizer
print('[1/5] Loading tokenizer...')
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# Find or load REPEAT_S2
REPEAT_S2 = None
if os.path.exists(CONFIG_FILE):
    REPEAT_S2 = json.load(open(CONFIG_FILE))['REPEAT_S2']
    print(f'[2/5] Loaded REPEAT_S2={REPEAT_S2} from config')
else:
    FMT_CKPT = 'checkpoint-530'
    ckp = os.path.join(S1_DIR, FMT_CKPT)
    if not os.path.isdir(ckp):
        avail = sorted([n for n in os.listdir(S1_DIR) if n.startswith('checkpoint-') and n.split('-')[-1].isdigit()],
                       key=lambda x: int(x.split('-')[-1]))
        ckp = os.path.join(S1_DIR, avail[-1]) if avail else None
        FMT_CKPT = avail[-1] if avail else 'none'
    print(f'[2/5] Searching REPEAT_S2 at {FMT_CKPT} (target count5>={FMT_THRESHOLD})...')
    with open(EVAL_JSON,'r',encoding='utf-8') as f: edat = json.load(f)
    for rpt in range(1, 6):
        exs = []
        for q_text, ans_obj in RS2 * rpt:
            ans = json.dumps(ans_obj, ensure_ascii=False)
            ms = [{'role':'system','content':TOP5_S2},{'role':'user','content':'/no_think\n案情：\n'+q_text+'\n\n请严格输出 JSON。'}]
            p = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            exs.append({'prompt':p, 'completion':ans+tok.eos_token})
        ds = Dataset.from_list(exs).train_test_split(test_size=0.1, seed=42)
        s2t = ds['train']; s2e = ds['test'] if 'test' in ds else ds['eval']
        b = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True)
        m = PeftModel.from_pretrained(b, ckp, is_trainable=True)
        a = SFTConfig(output_dir='/tmp/s2_search', learning_rate=LR_S2, num_train_epochs=1.0,
            per_device_train_batch_size=BS, gradient_accumulation_steps=GAS,
            logging_steps=999, save_strategy='no',
            bf16=torch.cuda.is_available(), fp16=not torch.cuda.is_available(),
            max_length=768, packing=False, report_to='none',
            warmup_steps=5, lr_scheduler_type='cosine', completion_only_loss=True)
        t = SFTTrainer(model=m, args=a, train_dataset=s2t, eval_dataset=s2e, processing_class=tok)
        t.train()
        fd = '/tmp/s2_search_final'
        t.model.save_pretrained(fd); tok.save_pretrained(fd)
        del t, m, b; gc.collect(); torch.cuda.empty_cache()
        b = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True)
        m = PeftModel.from_pretrained(b, fd, is_trainable=False); m.eval()
        c5 = 0
        for it in edat[:100]:
            usr = '/no_think\n案情：\n'+it['question']+'\n\n请严格输出 JSON。'
            ms = [{'role':'system','content':TOP5_S2},{'role':'user','content':usr}]
            tx = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            ins = tok(tx, return_tensors='pt').to(m.device)
            ot = m.generate(**ins, max_new_tokens=256, do_sample=False,
                eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
            rw = tok.decode(ot[0][ins['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
            mt = re.search(r'\{.*\}', rw, re.S)
            if mt:
                try:
                    obj = json.loads(mt.group())
                    if isinstance(obj,dict) and isinstance(obj.get('articles'),list) and len(obj['articles'])==5:
                        c5 += 1
                except: pass
        print(f'    REPEAT_S2={rpt}: count5={c5}/100', 'PASS' if c5>=FMT_THRESHOLD else 'FAIL')
        del m, b; gc.collect(); torch.cuda.empty_cache()
        if c5 >= FMT_THRESHOLD:
            REPEAT_S2 = rpt
            with open(CONFIG_FILE, 'w') as f: json.dump({'REPEAT_S2':REPEAT_S2, 'count5':c5}, f)
            print(f'    -> Saved to {CONFIG_FILE}')
            break
    if REPEAT_S2 is None: REPEAT_S2 = 5; print(f'    Fallback to REPEAT_S2=5')

# Build Stage-2 data
print(f'[2/5] Building Stage-2 data (REPEAT_S2={REPEAT_S2})...')
examples = []
for q_text, ans_obj in RS2 * REPEAT_S2:
    ans = json.dumps(ans_obj, ensure_ascii=False)
    ms = [{'role':'system','content':TOP5_S2},{'role':'user','content':'/no_think\n案情：\n'+q_text+'\n\n请严格输出 JSON。'}]
    p = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    examples.append({'prompt':p, 'completion':ans+tok.eos_token})
ds = Dataset.from_list(examples).train_test_split(test_size=0.1, seed=42)
s2_train = ds['train']
s2_eval  = ds['test'] if 'test' in ds else ds['eval']
print(f'  {len(s2_train)} train, {len(s2_eval)} eval')

[1/5] Loading tokenizer...
[2/5] Loaded REPEAT_S2=2 from config
[2/5] Building Stage-2 data (REPEAT_S2=2)...
  18 train, 2 eval


In [2]:
# ===== 2-Base model eval + Eval helpers =====
ART_RE = r'《.+?》第[0-9０-９一二三四五六七八九十百千万零〇○两]+条'
CN_M = {'零':0,'〇':0,'○':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
CN_U = {'十':10,'百':100,'千':1000,'万':10000}
def c2i(t):
    if not t: return None
    t = str(t).strip().translate(str.maketrans('０１２３４５６７８９','0123456789'))
    if t.isdigit(): return int(t)
    tot = sec = num = 0
    for c in t:
        if c in CN_M: num = CN_M[c]
        elif c in CN_U:
            u = CN_U[c]
            if u == 10000: sec = (sec+(num or 0))*u; tot += sec; sec = num = 0
            else:
                if num == 0: num = 1
                sec += num*u; num = 0
        else: return None
    return tot+sec+num
def canon(a):
    if not isinstance(a, str): return ''
    x = a.strip(); x = re.sub(r'\s+','',x); x = x.replace('（','(').replace('）',')')
    x = x.translate(str.maketrans('０１２３４５６７８９','0123456789'))
    m = re.match(r'^(《.*?》)第([0-9一二三四五六七八九十百千万零〇○两]+)条$', x)
    if not m: return x
    l = re.sub(r'^《中华人民共和国','《', m.group(1))
    n = c2i(m.group(2))
    return f'{l}第{n}条' if n is not None else f'{l}第{m.group(2)}条'

@torch.no_grad()
def gen_top5(q, mdl):
    ms = [{'role':'system','content':TOP5_S2},{'role':'user','content':'/no_think\n案情：\n'+q+'\n\n请严格输出 JSON。'}]
    tx = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ins = tok(tx, return_tensors='pt').to(mdl.device)
    ot = mdl.generate(**ins, max_new_tokens=256, do_sample=True, repetition_penalty=1.05,
        eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    rw = tok.decode(ot[0][ins['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    mt = re.search(r'\{.*\}', rw, re.S)
    preds = []
    if mt:
        try:
            obj = json.loads(mt.group())
            if isinstance(obj,dict) and isinstance(obj.get('articles'),list):
                preds = [canon(x) for x in obj['articles'] if canon(x)]
        except: pass
    if not preds:
        preds = [canon(x) for x in re.findall(ART_RE, rw)[:5] if canon(x)]
    return {'raw':rw, 'pred':preds}

with open(EVAL_JSON,'r',encoding='utf-8') as f: eval_data = json.load(f)
print(f'[3/5] Eval: {len(eval_data)} questions')

# Base model eval — save per-sample + summary
BASE_OUT = os.path.join(RESULTS_DIR, 'base_local_per_sample.json')
if os.path.exists(BASE_OUT):
    bm = json.load(open(BASE_OUT))['summary']
    print(f'  Base (cached): P@5={bm["P@5"]:.4f} R@5={bm["R@5"]:.4f} F1@5={bm["F1@5"]:.4f}')
else:
    print('  Running base eval (excluding Stage-2 questions)...')
    edat_base = [it for it in eval_data if it['question'] not in S2_QUESTIONS]
    print(f'  {len(eval_data)} total -> {len(edat_base)} after excluding S2')
    bmodel = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True)
    bmodel.eval()
    brs = []
    for it in tqdm(edat_base, desc='base'):
        g = gen_top5(it['question'], bmodel)
        ps = set(g['pred']); gs = set(canon(a) for a in it['articles'] if canon(a))
        t = len(ps&gs); p = t/max(len(ps),1); r = t/max(len(gs),1)
        f = 2*p*r/(p+r) if p+r>0 else 0.0
        brs.append({'id':it['id'],'q':it['question'],'gold':list(gs),'pred':g['pred'],
                     'P':round(p,6),'R':round(r,6),'F1':round(f,6),'raw':g['raw']})
    del bmodel; gc.collect(); torch.cuda.empty_cache()
    N = len(brs)
    bm = {'model':'base','samples':N,'P@5':round(sum(r['P'] for r in brs)/N,4),
          'R@5':round(sum(r['R'] for r in brs)/N,4),'F1@5':round(sum(r['F1'] for r in brs)/N,4)}
    print(f'  Base: P@5={bm["P@5"]:.4f} R@5={bm["R@5"]:.4f} F1@5={bm["F1@5"]:.4f}')
    with open(BASE_OUT,'w',encoding='utf-8') as f:
        json.dump({'summary':bm,'per_sample':brs}, f, ensure_ascii=False, indent=2)
    print(f'  Saved per-sample to {BASE_OUT}')

def eval_adapter(path, lab):
    gc.collect(); torch.cuda.empty_cache()
    b = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True)
    m = PeftModel.from_pretrained(b, path, is_trainable=False); m.eval()
    rs = []; tp = pr = gt = 0
    for it in tqdm(eval_data, desc=lab):
        g = gen_top5(it['question'], m)
        ps = set(g['pred']); gs = set(canon(a) for a in it['articles'] if canon(a))
        t = len(ps&gs); p = t/max(len(ps),1); r = t/max(len(gs),1)
        f = 2*p*r/(p+r) if p+r>0 else 0.0
        tp += t; pr += len(ps); gt += len(gs)
        rs.append({'id':it['id'],'q':it['question'],'gold':list(gs),'pred':g['pred'],'P':round(p,6),'R':round(r,6),'F1':round(f,6),'raw':g['raw']})
    mp = sum(r['P'] for r in rs)/len(rs); mr = sum(r['R'] for r in rs)/len(rs); mf = sum(r['F1'] for r in rs)/len(rs)
    del m, b; gc.collect(); torch.cuda.empty_cache()
    s = int(lab.split('-')[-1]) if '-' in lab else 0
    return {'label':lab,'step':s,'P@5':round(mp,4),'R@5':round(mr,4),'F1@5':round(mf,4),'ps':rs}

[3/5] Eval: 362 questions
  Running base eval (excluding Stage-2 questions)...
  362 total -> 361 after excluding S2


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

base:   0%|          | 0/361 [00:00<?, ?it/s]

  Base: P@5=0.0853 R@5=0.3631 F1@5=0.1353
  Saved per-sample to /root/autodl-tmp/civil_ft/results/base_local_per_sample.json
